# Phase 3, step 1: the smoke test (one policy, 10 motions)

Before training on the whole dataset, check the plumbing on a small library:
**10 training motions joined into one library, trained with the course's tracking task and
config unchanged** (`Mjlab-Tracking-Flat-Pupper-v3`, 4096 envs, 3,000 iterations, ~15 min on a G4).

**Done when** the one policy tracks all 10: `play_motion --checkpoint` on each motion's own npz
says HOLDS UP (worst error ≤ 3 cm and ≤ 10°), better than plain replay. Three validation motions it
never saw are scored too, as a first peek. With only 10 training motions, nobody expects those to
track well yet.

Needs a **G4 runtime** (Runtime → Change runtime type). Checkpoints go to Drive and W&B.
**Nothing here moves the robot.** A new trained policy needs a TA's OK before it ever goes on the robot.

In [ ]:
# @title Setup: pupper-mjlab (pinned), kflib, Drive (about 2 minutes)
import os, sys, subprocess
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"
BRANCH = "final-project-kflib"  # @param {type:"string"}
DATASET_NAME = "v1"  # @param {type:"string"}
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: switch to a G4 runtime")

if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
DATASET = ROOT / "datasets" / DATASET_NAME
RUNS = ROOT / "runs"  # checkpoints are written here (and to W&B), so a disconnect loses nothing
assert (DATASET / "manifest.csv").exists(), f"no dataset at {DATASET}"

In [ ]:
# @title Log into W&B (the key comes from Colab Secrets, never from this notebook)
# Add WANDB_API_KEY in Colab's key icon (left bar), notebook access on. Use the W&B
# account your Pupper uses. Without the secret, wandb asks for the key interactively.
import wandb
try:
  from google.colab import userdata
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:  # noqa: BLE001
  pass
wandb.login()
print("W&B entity:", wandb.Api().default_entity)

In [ ]:
# @title Pick 10 training motions (and 3 unseen), build the library
import json
from kflib.dataset import Dataset, build_library
ds = Dataset(DATASET)
orig = [r for r in ds.rows() if not r["aug"]]
train = [r for r in orig if r["split"] == "train"]
val = [r for r in orig if r["split"] == "val"]

pick = [r for r in train if r["source"] == "course"][:1]  # sit
for cat in ("precise", "mood", "dog_action", "sequence", "pose_sweep"):
  pick += [r for r in train if r["source"] == "gemini" and r["kind"] == cat][:1]
proc = [r for r in train if r["source"] == "procedural"]
pick += [r for r in proc if r["verdict"] == "WOBBLES"][:2] + [r for r in proc if r["verdict"] == "HOLDS UP"][:2]
unseen = [r for r in val if r["source"] == "gemini"][:2] + [r for r in val if r["source"] == "procedural"][:1]
assert len(pick) == 10 and len(unseen) == 3, (len(pick), len(unseen))
for r in pick + unseen:
  print(f"{r['split']:5} {r['id']:16} {r['source']:10} {r['verdict']:9} {r['duration_s']:>5} s  {r['request'][:60]}")

LIB_NAME = "pgt_smoke10"
LIB_DIR = ROOT / "libraries"
LIB_DIR.mkdir(parents=True, exist_ok=True)
LIB = LIB_DIR / f"{LIB_NAME}.npz"
library = build_library({r["id"]: ds.load_motion(r["id"]) for r in pick}, LIB)
(LIB_DIR / f"{LIB_NAME}.json").write_text(json.dumps(dict(dataset=str(DATASET), ids=[r["id"] for r in pick],
                                                          unseen=[r["id"] for r in unseen], clips=library.clips), indent=1))
print(f"\nlibrary: {len(library.root_pos)} frames = {len(library.root_pos) / 50:.0f} s ->", LIB)

In [ ]:
# @title Plain replay of the whole library (a sanity check before spending GPU time)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {LIB} 2>/dev/null | tail -4

In [ ]:
# @title Upload the library to W&B
!python -m mjlab.tasks.tracking.config.pupper.upload_motion {LIB} --name {LIB_NAME} 2>/dev/null | tail -2
MOTION_ARTIFACT = f"{wandb.Api().default_entity}/mjlab/{LIB_NAME}"
print("motion artifact:", MOTION_ARTIFACT)

## Train (~15 minutes)

The course's task and settings, unchanged: only the motion (our library), the run name and where
checkpoints go. Watch `Metrics/motion/error_*` (lower is better) and `Episode_Reward/motion_*`
(closer to 1 is better) on the W&B link it prints.

In [ ]:
# @title Train the smoke-test policy
%cd /content/pupper-mjlab
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
import mjlab.tasks  # registers the tasks

TASK = "Mjlab-Tracking-Flat-Pupper-v3"
env_cfg = load_env_cfg(TASK)
env_cfg.scene.num_envs = 4096
agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = 3000
agent_cfg.logger = "wandb"
agent_cfg.run_name = LIB_NAME
launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg, registry_name=MOTION_ARTIFACT, log_root=str(RUNS)))

In [ ]:
# @title The checkpoint
run_dirs = sorted((RUNS / "pupper_tracking").glob(f"*_{LIB_NAME}"), key=lambda p: p.stat().st_mtime)
RUN_DIR = run_dirs[-1]
CHECKPOINT = max(RUN_DIR.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("run:", RUN_DIR.name, "| checkpoint:", CHECKPOINT.name)

## Score: does one policy track all 10?

Each motion's own npz, played by plain replay and by the policy (4 at a time, ~10-15 min). The
three unseen ones are the first peek at zero-shot; don't read much into them yet.

In [ ]:
# @title Score the 10 training motions and the 3 unseen ones
from kflib.score import score, compare
motions = [dict(r, npz=str(ds.motions / f"{r['id']}.npz")) for r in pick + unseen]
rows = score(motions, CHECKPOINT, out_csv=RUN_DIR / "scores.csv", workers=4)
print()
table = compare(rows)
for t in table:
  print(f"{t['split']:5} {t['id']:16} plain: {t['plain']:28} policy: {t['policy']:28} {'tracks' if t['tracks'] else ''}")
n = sum(t["tracks"] for t in table if t["split"] == "train")
print(f"\ntraining motions tracked (HOLDS UP under the policy): {n}/10"
      f" | unseen: {sum(t['tracks'] for t in table if t['split'] == 'val')}/3")

In [ ]:
# @title (Optional) Watch the policy on one motion (public link; stop the cell when done)
WATCH = pick[0]["id"]  # @param {type:"string"}
!python -m mjlab.tasks.tracking.config.pupper.play_motion {ds.motions / (WATCH + '.npz')} --checkpoint {CHECKPOINT} --viewer viser --share True --loops 2